# Functions — Senior Python Interview Practice\n\nFour interview-style exercises spanning implementation, trade-offs, and production concerns.\n\n**How to use:** attempt each prompt first, then run and critique the reference solution. Discuss trade-offs aloud as you would in a senior-level interview.


## 1. Retry with backoff\n\n### Problem statement\nWrite a decorator factory that retries a function and uses exponential backoff; re-raise the last exception.


In [ ]:
import time
from functools import wraps

def retry(attempts=3, base_delay=0.1, sleep=time.sleep):
    def decorate(fn):
        @wraps(fn)
        def wrapped(*args, **kwargs):
            for attempt in range(attempts):
                try: return fn(*args, **kwargs)
                except Exception:
                    if attempt == attempts - 1: raise
                    sleep(base_delay * 2 ** attempt)
        return wrapped
    return decorate


### Complexity\n- **Time:** O(attempts × function cost)\n- **Space:** O(1)\n\n### Interview tip\nInject `sleep` for deterministic tests and never retry every exception blindly in production.\n\n### Follow-up questions\n- Add jitter, a retry predicate, deadlines, and tracing.


## 2. Memoized Fibonacci\n\n### Problem statement\nUse a closure to memoize a recursive Fibonacci implementation.


In [ ]:
def memoized_fib():
    cache = {0: 0, 1: 1}
    def fib(n):
        if n < 0: raise ValueError('n must be non-negative')
        if n not in cache: cache[n] = fib(n - 1) + fib(n - 2)
        return cache[n]
    return fib

fib = memoized_fib(); fib(10)


### Complexity\n- **Time:** O(n) for first call\n- **Space:** O(n)\n\n### Interview tip\nDiscuss closure state, cache lifetime, and the recursion limit.\n\n### Follow-up questions\n- When should `functools.lru_cache` replace this?


## 3. Chunk an iterable\n\n### Problem statement\nYield fixed-size lists from any iterable without materializing it all.


In [ ]:
from itertools import islice

def chunks(iterable, size):
    if size <= 0: raise ValueError('size must be positive')
    it = iter(iterable)
    while batch := list(islice(it, size)):
        yield batch

list(chunks(range(10), 3))


### Complexity\n- **Time:** O(n)\n- **Space:** O(size)\n\n### Interview tip\nShow awareness that iterator inputs are consumed once.\n\n### Follow-up questions\n- Should the final partial chunk be emitted, padded, or dropped?


## 4. Callable strategy\n\n### Problem statement\nBuild a scorer factory that returns a function combining a score function and a threshold.


In [ ]:
def make_acceptor(score_fn, threshold):
    def accepts(item):
        return score_fn(item) >= threshold
    return accepts

accepts_long = make_acceptor(len, 5)
accepts_long('agentic')


### Complexity\n- **Time:** O(score function)\n- **Space:** O(1)\n\n### Interview tip\nExplain why passing behavior as a value makes policies easy to test and swap.\n\n### Follow-up questions\n- How would you make the policy configurable and observable?
